# 📝 가설검정·회귀 과제 LV2(응용) — 종별 체중 검정·회귀

> 가설검정과 회귀분석을 **이어서 조합**하는 문제입니다. 팔머 펭귄(`penguins`) 데이터로 가정 점검 → 분산분석 → 사후검정 → 카이제곱 → 단순·다중 회귀 → 잔차진단을 차례로 연습합니다.

## 풀이 방법
1. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
2. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요. (**그래프·서술 문제는 자가채점이 없어요** — 그래프는 완성 그림과 같은 모양으로 그리고, 서술은 관찰을 직접 적습니다.)
3. 막히면 `힌트` 를 펼쳐 보세요.

- 데이터는 `data/penguins.csv` 를 씁니다(종·서식지·부리·물갈퀴·체중·성별).
- **penguins 에는 결측치가 있습니다** — 각 문제에서 쓰는 열에 `dropna` 로 결측 행을 먼저 제거하세요(문제마다 명시).
- **문제마다 원본 CSV 를 다시 불러와** 시작하세요(앞 문제의 가공이 뒤 문제에 섞이지 않도록).
- 문제 **1·2·9** 에는 **서술 셀**이 있습니다 — 자가채점/그래프 아래 markdown 셀에 관찰을 직접 적으세요.

화이팅!

아래 셀을 먼저 실행해 검정·회귀 라이브러리와 한글 폰트를 준비하세요.

In [ ]:
# [제공 코드] 통계 검정·회귀에 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')   # pingouin 의 사소한 경고를 숨겨 출력을 깔끔하게
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg                                  # ★ 검정 주력: 검정+효과크기를 한 표로
import scikit_posthocs as sp                          # 비모수 사후검정 Dunn (pingouin 에 없음)
import statsmodels.formula.api as smf                  # 회귀 (smf.ols) — pingouin 미지원이라 유지

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={'axes.unicode_minus': False})

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치 요약을, `describe(exclude='number')` 로 범주형 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치·범주 요약
df = pd.read_csv('data/penguins.csv')
print('행·열 크기:', df.shape)
print('\n[앞 5행] head()'); display(df.head())
print('\n[열·자료형·결측] info()'); df.info()
print('\n[수치 요약] describe()'); display(df.describe())
print('\n[범주형 요약] describe(exclude="number")'); display(df.describe(exclude='number'))

## 1. 데이터를 살펴보고 관찰 적기 (서술)
**배경**: 분석을 시작하기 전, 위 `데이터 살펴보기` 결과를 바탕으로 이 데이터가 어떤 데이터인지 스스로 정리해 봅니다.

**요구사항**:
- 위 `# [제공 코드]`(head·info·describe) 실행 결과를 보고, 아래 **서술 셀**에 관찰을 **2~3문장**으로 적으세요.
- 다음을 담으면 좋아요: 몇 개 종의 몇 개 행인지, 어떤 열이 수치/범주인지, 어느 열에 결측치가 있는지, 종에 따라 체중(`body_mass_g`)이 대략 어떻게 다른지.

**관찰 (서술)**

*(여기에 2~3문장으로 데이터 관찰을 적으세요 — 종 수·행 수, 수치/범주 열, 결측 위치, 종별 체중 차이)*

## 2. 정규성·등분산 점검과 검정 선택 (가정 점검 + 서술)
**배경**: 세 종의 체중 평균을 비교하기 전에, 어떤 검정을 써야 할지 정하려면 **정규성**(Shapiro-Wilk)과 **등분산**(Levene)을 먼저 확인해야 합니다.

**요구사항**:
- `data/penguins.csv` 를 읽어 `body_mass_g`·`species` 열의 **결측 행을 제거**하세요(`dropna(subset=['body_mass_g', 'species'])`).
- 종별로 체중을 나눠 `adelie`·`chinstrap`·`gentoo` 에 담으세요(예: `adelie = df[df['species'] == 'Adelie']['body_mass_g']`).
- `pg.normality` 로 각 종 체중의 정규성을 검정해 **pval**(`['pval'].iloc[0]`)을 `shapiro_p_adelie`·`shapiro_p_chinstrap`·`shapiro_p_gentoo` 에 담으세요.
- `pg.homoscedasticity(data=df, dv='body_mass_g', group='species')` 로 등분산(Levene)을 검정해 **W**(`['W'].iloc[0]`, 검정통계량)를 `levene_stat`, **pval**(`['pval'].iloc[0]`)을 `levene_p` 에 담으세요.
- p값은 소수 **넷째 자리**, 검정통계량은 소수 **셋째 자리**까지 비교합니다.
- 자가채점 아래 **서술 셀**에, 정규성·등분산 결과를 근거로 어떤 분산분석을 쓸지와 그 한계를 **2문장 이상** 적으세요.

**예시**
```
round(shapiro_p_adelie, 4)    → 0.0324
round(shapiro_p_chinstrap, 4) → 0.5605
round(shapiro_p_gentoo, 4)    → 0.2336
round(levene_stat, 3)         → 5.12
round(levene_p, 4)            → 0.0064
```
<details><summary>힌트</summary>

```text
접근방법:
- 종별로 체중을 나눈 뒤, 각 종에 pg.normality 로 정규성을, pg.homoscedasticity 로 등분산(Levene)을 확인한다.
- pg.normality 는 결과 표에서 pval 을, pg.homoscedasticity 는 W(검정통계량)·pval 을 꺼낸다.

세부구현:
1. penguins 를 불러와 체중·종의 결측 행을 제거한다
2. 종 이름으로 걸러 세 종의 체중 시리즈를 각각 만든다
3. 각 종에 pg.normality 를 돌려 ['pval'].iloc[0] 를 담는다
4. pg.homoscedasticity(data=df, dv='body_mass_g', group='species') 로 W·pval 을 담는다
5. 두 검정 결과를 근거로 어떤 분산분석이 적절한지 서술한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(shapiro_p_adelie - 0.0324) < 0.01
assert abs(shapiro_p_chinstrap - 0.5605) < 0.01
assert abs(shapiro_p_gentoo - 0.2336) < 0.01
assert abs(levene_stat - 5.12) < 0.01
assert abs(levene_p - 0.0064) < 0.01, 'levene(adelie, chinstrap, gentoo) 순서로 세 종을 함께 넣으세요'
print("✅ 문제2 통과!")

**검정 선택 (서술)**

*(여기에 2문장 이상으로 서술하세요 — 정규성·등분산 검정 결과, 그에 따라 어떤 분산분석을 쓸지, 그리고 그 한계)*

## 3. 종별 체중 차이 — 일원분산분석 + 효과크기 η² (검정)
**배경**: 앞에서 확인한 가정을 바탕으로, 세 종의 체중 평균이 서로 다른지 **일원분산분석(ANOVA)** 으로 검정하고 효과크기 **η²**(집단이 설명하는 분산 비율)까지 구합니다.

**요구사항**:
- `data/penguins.csv` 를 읽어 `body_mass_g`·`species` 결측 행을 제거해 `df` 에 담으세요(`dropna`).
- `pg.anova(data=df, dv='body_mass_g', between='species', detailed=True)` 로 일원분산분석 표를 만들어 `aov` 에 담고 `display(aov)` 하세요(긴 형태 — 값 열은 `dv`, 그룹 열은 `between`).
- 결과 표 첫 행에서 **F**(`aov['F'].iloc[0]`)를 `anova_f`, **p_unc**(`aov['p_unc'].iloc[0]`)을 `anova_p` 에 담으세요.
- **효과크기 η²** 는 결과 표의 **np2** 열(`aov['np2'].iloc[0]`)을 그대로 `eta_squared` 에 담으세요 — 일원분산분석에서 부분 η²(`np2`)가 곧 η² 입니다.
- **(효과크기 정의 손계산)** η² 의 정의는 **집단간제곱합 ÷ 총제곱합** 입니다. 분산분석 표(`aov`)의 **SS** 열에서 집단(첫 행, `aov['SS'].iloc[0]`)의 제곱합을 `ss_between`, 잔차(둘째 행 `Within`, `aov['SS'].iloc[1]`)의 제곱합을 `ss_within` 에 담고, `eta_hand = ss_between / (ss_between + ss_within)` 으로 **직접 계산**해 pingouin 의 `eta_squared`(=`np2`)와 **거의 같은지** 확인하세요(총제곱합 = 집단간 + 잔차).
- 검정통계량·η² 는 소수 **셋째 자리**, p값은 소수 **넷째 자리**까지 비교합니다.

**예시**
```
round(anova_f, 3)     → 343.626
round(anova_p, 4)     → 0.0
round(eta_squared, 3) → 0.67
round(eta_hand, 3)    → 0.67     # 손계산 = np2 와 일치
```
<details><summary>힌트</summary>

```text
접근방법:
- pg.anova 는 긴 형태(dv·between) 한 번으로 F·p·효과크기(np2)를 한 표로 준다.
- η² 는 정의상 집단간제곱합을 총제곱합(=집단간+잔차)으로 나눈 값이다 — 표의 SS 열로 직접 확인할 수 있다.

세부구현:
1. penguins 를 불러와 체중·종 결측을 제거한다
2. pg.anova(data=df, dv='body_mass_g', between='species', detailed=True) 로 표 aov 를 만든다
3. aov['F']·['p_unc']·['np2'] 의 .iloc[0] 를 anova_f, anova_p, eta_squared 에 담는다
4. aov['SS'] 의 첫 행·둘째 행을 ss_between, ss_within 에 담아 eta_hand = ss_between/(ss_between+ss_within) 로 계산한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(anova_f - 343.626) < 0.01
assert abs(anova_p - 0.0) < 0.01
assert abs(eta_squared - 0.67) < 0.01
assert abs(eta_hand - 0.67) < 0.01, 'η² = 집단간제곱합 ÷ 총제곱합 이에요'
assert abs(eta_hand - eta_squared) < 0.001, '손계산 η² 가 pingouin np2 와 일치해야 해요'
print("✅ 문제3 통과!")

## 4. 어느 종끼리 다른가 — Tukey 사후검정 (표)
**배경**: 분산분석은 '적어도 하나가 다르다'까지만 알려 줍니다. **어느 종 쌍**이 실제로 다른지 알려면 **Tukey HSD 사후검정**으로 모든 종 쌍을 비교해야 합니다.

**요구사항**:
- `data/penguins.csv` 를 읽어 `body_mass_g`·`species` 결측 행을 제거해 `df` 에 담으세요(`dropna`).
- `pg.pairwise_tukey(data=df, dv='body_mass_g', between='species')` 로 사후검정 표를 만들어 `tukey_result` 에 담고 `display(tukey_result)` 하세요.
- **유의하게 다른 종 쌍의 개수**를 `n_significant` 에 담으세요(`int((tukey_result['p_tukey'] < 0.05).sum())`). `p_tukey` 가 0.05 미만인 쌍이 유의하게 다른 쌍입니다.

**예시**
```
n_significant → 2   (전체 3쌍 중 2쌍이 유의하게 다름)
tukey_result: A·B·mean_A·mean_B·diff·se·T·p_tukey·hedges 열 (쌍별 효과크기 hedges 포함)
```
<details><summary>힌트</summary>

```text
접근방법:
- pg.pairwise_tukey 는 모든 집단 쌍을 한꺼번에 비교한 표(DataFrame)를 주며 다중검정 보정과 효과크기(hedges)까지 포함한다.
- p_tukey 열이 0.05 미만인 쌍이 통계적으로 유의하게 다른 쌍이다.

세부구현:
1. penguins 를 불러와 결측 제거해 df 에 담는다
2. pg.pairwise_tukey(data=df, dv='body_mass_g', between='species') 로 표 tukey_result 를 만들어 화면에 보여 준다
3. (tukey_result['p_tukey'] < 0.05).sum() 을 int 로 감싸 n_significant 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_significant == 2, 'p_tukey < 0.05 인 종 쌍의 개수예요 (Gentoo 가 다른 두 종과 다름)'
print("✅ 문제4 통과!")

## 5. 가정이 깨졌다면 — Kruskal-Wallis (ANOVA 의 비모수 짝)
**배경**: 문제 2 에서 정규성·등분산을 점검하고, 문제 3 에서 **일원분산분석(ANOVA)** 으로 종별 체중 차이를 확인했습니다. 그런데 ANOVA 는 정규성·등분산을 전제합니다. 가정이 흔들릴 때 3집단 이상을 비교하는 비모수 대안이 **Kruskal-Wallis**(`pg.kruskal`)예요. 값 대신 **순위**를 쓰므로 가정에 덜 민감합니다. 같은 데이터에 돌려 보고 ANOVA 결론과 견줍니다.

**요구사항**:
- `data/penguins.csv` 를 읽어 `df` 에 담고, `body_mass_g` 의 **결측을 제거**하세요.
- `pg.kruskal(data=..., dv='body_mass_g', between='species')` 로 결과 표를 만들어 `display` 하세요.
- 그 표에서 검정통계량 `H` 를 `h_stat`, p-value `p_unc` 를 `kruskal_p` 에 담으세요.
- 종별 체중 **중앙값**을 `species_median`(Series, 종 이름이 인덱스)에 담으세요. Kruskal 은 평균이 아니라 순위를 보므로 중앙값이 짝이 되는 요약값입니다.
- **사후검정**: Kruskal 도 ANOVA 처럼 "어딘가 다르다" 까지만 말합니다. 어느 쌍이 다른지는 **Dunn 검정**으로 봅니다 — `sp.posthoc_dunn(df, val_col='body_mass_g', group_col='species', p_adjust='holm')` 로 쌍별 p 표를 만들어 `dunn` 에 담고 `display` 하세요(`import scikit_posthocs as sp`). 쌍이 여럿이라 **다중검정 보정**(`p_adjust='holm'`)이 필요합니다.
- 그 표에서 **유의한 쌍(p < 0.05)의 개수**를 `n_sig_dunn` 에 담으세요. 표가 대칭이고 대각선이 1.0 이므로 `int(((dunn < 0.05).sum().sum()) / 2)` 로 셉니다.
- `h_stat` 은 소수 둘째, `kruskal_p` 는 0 에 아주 가까운 값이라 `< 1e-40` 인지로 채점됩니다.

**예시**
```
round(h_stat, 2)          →  217.60
kruskal_p                 →  5.61e-48  (0 에 가까움)
species_median['Gentoo']  →  5000.0
n_sig_dunn                →  2        (문제 4 의 Tukey 와 같은 개수)
```
> **눈여겨볼 것**: 문제 3 의 ANOVA 도, 여기 Kruskal 도 **같은 결론**(종별 차이가 있다)에 이릅니다. 표본이 크고 차이가 뚜렷하면 두 방법이 잘 일치해요. 아래 서술에서 그 의미를 정리해 보세요.
<details><summary>힌트</summary>

```text
접근방법:
- 3집단 이상의 비모수 비교 함수에 데이터프레임·종속변수·집단변수를 이름으로 넘긴다.
- 순위 기반 검정이므로 요약값은 평균이 아니라 중앙값으로 본다.

세부구현:
1. read_csv 로 읽고 body_mass_g 결측을 제거한다(dropna 의 subset)
2. 비모수 3집단 검정 함수를 data·dv·between 인자로 호출해 결과 표를 받는다
3. 결과 표의 H 열과 p_unc 열 첫 값을 각각 h_stat·kruskal_p 에 담는다
4. species 로 묶어 body_mass_g 의 중앙값을 구해 species_median 에 담는다
5. 지문의 Dunn 함수로 쌍별 p 표를 만들어 dunn 에 담고, p<0.05 인 칸 수를 2로 나눠 n_sig_dunn 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(h_stat - 217.60) < 0.01
assert kruskal_p < 1e-40
assert abs(species_median['Gentoo'] - 5000.0) < 0.01
assert abs(species_median['Adelie'] - 3700.0) < 0.01
assert n_sig_dunn == 2, 'Dunn 표에서 p<0.05 인 쌍의 개수예요 (Gentoo 가 다른 두 종과 다름)'
print("✅ 문제5 통과!")

**관찰 (서술)**

*(여기에 1~2문장으로 서술하세요 — ANOVA(문제 3)와 Kruskal 이 같은 결론에 이른 것이 무엇을 뜻하는지, 그리고 어떤 상황이라면 Kruskal 쪽을 써야 하는지)*

## 6. 종과 서식지는 독립인가 — 카이제곱 + Cramér's V (검정)
**배경**: 펭귄의 **종(`species`)** 과 **서식지(`island`)** 가 서로 관련 있는지(독립이 아닌지) **카이제곱 독립성 검정**으로 확인하고, 관련의 세기를 효과크기 **Cramér's V** 로 잽니다.

**요구사항**:
- `data/penguins.csv` 를 읽어 `species`·`island` 결측 행을 제거하세요(`dropna(subset=['species', 'island'])` — 이 두 열은 원래 결측이 없어 행 수는 그대로 344 입니다).
- `pg.chi2_independence(data=df, x='species', y='island')` 로 카이제곱 독립성 검정을 실행하세요. 이 함수는 **(기대빈도표, 관측빈도표, 통계량표)** 세 개를 순서대로 돌려줍니다: `expected, observed, chi_stats = pg.chi2_independence(...)`. 관측 교차표는 `display(observed)` 로 확인하세요.
- 통계량표 `chi_stats` 에서 **표준 Pearson 검정 행**(`chi_stats[chi_stats['test'] == 'pearson']`)을 골라 `chi2`(→`chi2`), `pval`(→`chi_p`), `cramer`(→`cramers_v`) 를 꺼내세요. **Cramér's V 는 `cramer` 열로 자동 계산**됩니다(손계산 불필요).
- **사후분석**: 카이제곱은 "두 변수가 관련 있다" 까지만 말합니다. **어느 칸이 기대보다 두드러지는지**는 **조정된 잔차**로 봅니다 — `Table(observed.values).standardized_resids` 로 계산해 `observed` 와 같은 행·열 이름을 붙인 DataFrame `resid_df` 에 담고 `display` 하세요(`from statsmodels.stats.contingency_tables import Table`).
- **|조정된 잔차| > 2 인 칸의 개수**를 `n_strong` 에 담으세요(`int((resid_df.abs() > 2).sum().sum())`). 양수면 기대보다 **많다**, 음수면 **적다**는 뜻입니다.
- 검정통계량·Cramér's V 는 소수 **셋째 자리**, p값은 소수 **넷째 자리**까지 비교합니다.

**예시**
```
round(chi2, 3)      → 299.55
round(chi_p, 4)     → 0.0
round(cramers_v, 3) → 0.66
n_strong            → 8      # 9칸 중 8칸이 |잔차| > 2
```
> **읽는 법**: Gentoo × Biscoe 의 조정된 잔차가 **+14.1** 로 가장 큽니다 — Gentoo 는 Biscoe 섬에 **유독 몰려 있다**는 뜻입니다. Cramér's V = 0.66 이라는 강한 연관이 **구체적으로 어느 칸에서 오는지**를 잔차가 알려 줍니다.
<details><summary>힌트</summary>

```text
접근방법:
- 두 범주형 변수의 관련성은 pg.chi2_independence 로 확인한다 — (기대, 관측, 통계량) 세 표를 준다.
- 통계량표의 test=='pearson' 행이 표준 카이제곱이고, cramer 열이 효과크기다.

세부구현:
1. penguins 를 불러와 종·서식지 결측을 제거한다
2. expected, observed, chi_stats = pg.chi2_independence(data=df, x='species', y='island')
3. pearson = chi_stats[chi_stats['test'] == 'pearson'] 로 표준 행을 고른다
4. pearson['chi2']·['pval']·['cramer'] 의 .iloc[0] 를 chi2, chi_p, cramers_v 에 담는다
5. Table(관측표.values).standardized_resids 로 조정된 잔차를 구해 행·열 이름을 붙여 resid_df 에 담는다
6. 절댓값이 2를 넘는 칸 수를 세어 n_strong 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(chi2 - 299.55) < 0.01
assert abs(chi_p - 0.0) < 0.01
assert abs(cramers_v - 0.66) < 0.01, "통계량표 pearson 행의 cramer 열이 Cramér's V 예요"
assert n_strong == 8, '조정된 잔차의 절댓값이 2를 넘는 칸 수예요 (9칸 중 8칸)'
assert abs(resid_df.loc['Gentoo', 'Biscoe'] - 14.13) < 0.05, 'Gentoo x Biscoe 잔차가 가장 큽니다'
print("✅ 문제6 통과!")

## 7. 물갈퀴 길이로 체중 예측 — 단순 선형회귀 (검정 + 그래프)
**배경**: 물갈퀴 길이(`flipper_length_mm`)가 길수록 체중(`body_mass_g`)이 무거울까요? **단순 선형회귀**로 관계를 수식으로 요약하고, 회귀선을 산점도 위에 그려 봅니다.

### (1) 회귀 적합 — 계수·R²·p값 (자가채점 있음)
**요구사항**:
- `data/penguins.csv` 를 읽어 `body_mass_g`·`flipper_length_mm` 결측 행을 제거하세요(`dropna`).
- `smf.ols('body_mass_g ~ flipper_length_mm', data=df).fit()` 로 회귀모형을 적합해 `simple_model` 에 담으세요.
- 결정계수 `simple_r2 = simple_model.rsquared`, 물갈퀴 계수 `simple_coef = simple_model.params['flipper_length_mm']`, 그 계수의 p값 `simple_p = simple_model.pvalues['flipper_length_mm']` 를 각각 담으세요.
- 결정계수·계수는 소수 **셋째 자리**, p값은 소수 **넷째 자리**까지 비교합니다.

**예시**
```
round(simple_r2, 3)   → 0.759
round(simple_coef, 3) → 49.686
round(simple_p, 4)    → 0.0
```
<details><summary>힌트</summary>

```text
접근방법:
- smf.ols 에 '종속변수 ~ 독립변수' 식과 데이터를 넣고 fit 한다.
- 적합 결과 객체에서 rsquared, params, pvalues 로 값을 꺼낸다.

세부구현:
1. penguins 를 불러와 체중·물갈퀴 결측을 제거한다
2. ols 식 'body_mass_g ~ flipper_length_mm' 로 모형을 적합한다
3. rsquared 로 결정계수, params['flipper_length_mm'] 로 기울기, pvalues 로 p값을 꺼낸다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(simple_r2 - 0.759) < 0.01
assert abs(simple_coef - 49.686) < 0.01, '물갈퀴 1mm 증가 시 체중이 약 49.686g 늘어난다는 뜻이에요'
assert abs(simple_p - 0.0) < 0.01
print("✅ 문제7 통과!")

### (2) 회귀선 그리기 (그래프 — 자가채점 없음)
**요구사항**:
- 위에서 적합한 회귀식으로 **산점도 위에 회귀선**을 그립니다. `flipper_length_mm`(x)와 `body_mass_g`(y)의 산점도를 `sns.scatterplot` 으로, 회귀선(빨간 직선)을 `sns.lineplot` 으로 겹쳐 그리세요.
- 회귀선은 `simple_model.params` 의 절편·기울기로 만듭니다: `y = 절편 + 기울기 × x` (x 는 물갈퀴 최소~최대 구간).
- 결과 Axes 를 `ax` 에 저장하고 제목을 다세요. 아래 **완성 그래프**와 같은 모양이면 정답입니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv2_q7.png" width="560"/>
<details><summary>힌트</summary>

```text
접근방법:
- 산점도를 먼저 그리고, 회귀식으로 만든 직선을 그 위에 겹쳐 그린다.
- 직선의 x 는 물갈퀴 최소~최대를 촘촘히 나눈 값, y 는 절편+기울기*x 로 만든다.

세부구현:
1. 결측 제거한 데이터를 다시 준비하고 회귀모형을 적합한다
2. 새 figure 를 만들고 scatterplot 으로 물갈퀴(x)-체중(y) 산점도를 그린다
3. np.linspace 로 x 구간을 만들고 절편+기울기*x 로 y 를 계산한다
4. 그 직선을 빨간색으로 겹쳐 그리고 제목을 단다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

## 8. 변수를 더하면 나아질까 — 다중 선형회귀 (검정)
**배경**: 물갈퀴 길이에 **부리 길이(`bill_length_mm`)** 를 더하면 체중 예측이 더 좋아질까요? **다중 선형회귀**로 두 변수를 함께 넣고, 각 변수의 기여를 계수 p값과 **조정 결정계수(Adj R²)** 로 확인합니다.

**요구사항**:
- `data/penguins.csv` 를 읽어 `body_mass_g`·`flipper_length_mm`·`bill_length_mm` 결측 행을 제거하세요(`dropna`).
- `smf.ols('body_mass_g ~ flipper_length_mm + bill_length_mm', data=df).fit()` 로 적합해 `multi_model` 에 담으세요.
- 조정 결정계수 `multi_adj_r2 = multi_model.rsquared_adj`, 물갈퀴 계수 `coef_flipper` 와 그 p값 `p_flipper`, 부리 계수 `coef_bill` 와 그 p값 `p_bill` 을 각각 꺼내세요.
- 조정 결정계수·계수는 소수 **셋째 자리**, p값은 소수 **넷째 자리**까지 비교합니다.

**예시**
```
round(multi_adj_r2, 3) → 0.759
round(coef_flipper, 3) → 48.145      round(p_flipper, 4) → 0.0
round(coef_bill, 3)    → 6.047       round(p_bill, 4)    → 0.2438
```
<details><summary>힌트</summary>

```text
접근방법:
- ols 식에 독립변수를 + 로 이어 두 개 이상 넣으면 다중회귀가 된다.
- 조정 결정계수는 변수 개수를 벌점으로 반영한 R² 로 rsquared_adj 로 꺼낸다.

세부구현:
1. 세 열의 결측을 제거한다
2. ols 식 'body_mass_g ~ flipper_length_mm + bill_length_mm' 로 적합한다
3. rsquared_adj 로 조정 R², params/pvalues 에서 각 변수의 계수와 p값을 꺼낸다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(multi_adj_r2 - 0.759) < 0.01
assert abs(coef_flipper - 48.145) < 0.01
assert abs(p_flipper - 0.0) < 0.01
assert abs(coef_bill - 6.047) < 0.01
assert abs(p_bill - 0.2438) < 0.01, '부리 길이 계수는 유의하지 않아요(p>0.05)'
print("✅ 문제8 통과!")

## 9. 회귀가 타당한가 — 잔차 진단 (그래프)
**배경**: 선형회귀가 성립하려면 **LINE 네 가정**(**선형성**·**독립성**·**정규성**·**등분산성**)이 필요하고, 이를 **잔차**로 점검합니다. 이 문제에서는 그중 **세 가지를 두 그림으로** 봅니다 — 잔차가 특정 패턴 없이 0 주변에 고르게 흩어지고(**선형성·등분산성**), 그 값들이 대체로 정규분포를 따라야(정규성) 합니다. **잔차 vs 적합값**과 **잔차 Q-Q Plot** 을 나란히 그려 두 가정을 눈으로 진단합니다.

**요구사항**:
- `data/penguins.csv` 를 읽어 `body_mass_g`·`flipper_length_mm` 결측 행을 제거하고, 문제 7과 같은 단순회귀(`body_mass_g ~ flipper_length_mm`)를 적합해 `simple_model` 에 담으세요.
- `plt.subplots(1, 2, figsize=(12, 5))` 로 서브플롯 두 개(`ax1`·`ax2`)를 만드세요.
- **왼쪽(`ax1`)**: x=적합값(`simple_model.fittedvalues`), y=잔차(`simple_model.resid`)의 **산점도**를 그리고, 잔차 0 위치에 빨간 **수평 점선**을 그으세요(`ax1.axhline(0, color='red', linestyle='--')`).
- **오른쪽(`ax2`)**: 잔차의 **Q-Q Plot** 을 `pg.qqplot(simple_model.resid, dist='norm', ax=ax2)` 로 그리세요. 점들이 대각선(빨간 기준선)에 가까울수록 잔차가 정규분포에 가깝다는 뜻입니다.
- 두 Axes 에 각각 제목을 다세요. 자가채점은 없습니다 — 아래 **완성 그래프**와 같은 모양이면 정답입니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv2_q9.png" width="760"/>
<details><summary>힌트</summary>

```text
접근방법:
- 서브플롯 두 개를 나란히 만들어 왼쪽엔 적합값-잔차 산점도, 오른쪽엔 잔차 Q-Q Plot 을 그린다.
- 왼쪽 점들이 기준선 위아래로 패턴 없이 흩어져 있으면 등분산·선형 가정이, 오른쪽 점들이 대각선에 가까우면 정규성 가정이 대체로 만족된 것이다.

세부구현:
1. 결측 제거 후 단순회귀 모형을 다시 적합한다
2. subplots(1, 2, ...) 로 ax1, ax2 두 축을 만든다
3. ax1 에 fittedvalues(x)-resid(y) 산점도를 그리고 axhline 으로 0 위치에 빨간 수평 점선을 긋는다
4. ax2 에 pg.qqplot(잔차, dist='norm', ax=ax2) 로 Q-Q Plot 을 그린다
5. 두 축에 각각 제목을 단다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

## 10. 인사이트 리포트 (종합 서술)
**배경**: 지금까지의 회귀 결과를 근거로 "체중을 예측할 때 어떤 변수가 중요한가"를 한 문단으로 정리합니다.

**요구사항**:
- 단순회귀(문제 7)와 다중회귀(문제 8)의 결과를 근거로 들며, **어떤 변수가 체중 예측에 기여하는지**를 **3문장 이상**으로 아래 서술 셀에 적으세요.
- 다음을 포함하면 좋아요: 물갈퀴 길이의 설명력(R²·계수 p값), 부리 길이를 더했을 때 조정 R²·계수 p값의 변화, 그리고 상관·회귀는 인과를 증명하지 않는다는 점.

**인사이트 리포트 (서술)**

*(여기에 3문장 이상으로 서술하세요 — 물갈퀴 길이의 설명력, 부리 길이 추가 효과, 어떤 변수가 핵심인지, 그리고 상관≠인과)*